# Diamond Clarity Classification

## Objective

Predict diamond clarity family using physical, categorical, and pricing information.

The original clarity grades are grouped into five ordered families:

**I → SI → VS → VVS → IF**

Three models are compared:

- Artificial Neural Network
- Random Forest
- XGBoost

Three experiments are performed:

1. Physical and categorical features without price
2. Add raw price
3. Add price-relative features and use the final model configurations

Experiments are compared using validation data.

The test set remains untouched until the final model has been selected.

The primary model-selection metric is **Macro F1** because the clarity
families are imbalanced.

**Scope:** This notebook and `src/classification/` are for clarity classification only. Regression uses a separate workflow.


## Data preparation

The implementation has moved to `../src/classification/`:

- `cleaning.py`: remove duplicates, nonpositive dimensions, and geometry outliers using the original 3 × IQR rule; validate columns and map clarity to **I → SI → VS → VVS → IF**.
- `feature_engineering.py`: physical measurements and ratios, raw price, and price relative to carat, volume, face area, and face diagonal.
- `pipeline.py`: one shared stratified **70% / 15% / 15%** train/validation/test split; median numeric imputation, standard scaling, and categorical imputation plus one-hot encoding.

Imputers, scalers, and encoders are fitted on training rows only. All three model types use the same scaled inputs, matching the original implementation. The original geometry-outlier bounds are still calculated before splitting.

The three feature configurations, model settings, validation comparison, and final test evaluation are preserved below. Cleaning now removes duplicates cumulatively and no longer reads `clean_df` before assignment, so rerun the experiments to obtain updated results.

DataFrames, statistics, distribution plots, and correlation matrices are displayed below. The same datasets are exported to `data/processed/classification/`.


In [ ]:
from pathlib import Path
import sys
import random

# Locate Diamond from either the project root or a notebook subfolder.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "src" / "classification" / "pipeline.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from the Diamond project or a subfolder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.classification import CLARITY_CLASSES, RANDOM_STATE, prepare_classification_data

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.utils.class_weight import compute_sample_weight

from xgboost import XGBClassifier

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

from src.classification.diagnostics import plot_top_features
from src.classification.evaluation import evaluate_model
from src.classification.model import build_ann


In [ ]:
# All cleaning, feature engineering, splitting, and transformation live in Python modules.
prepared = prepare_classification_data(PROJECT_ROOT / "data" / "raw" / "diamonds.csv")
y_train, y_valid, y_test = prepared["y_train"], prepared["y_valid"], prepared["y_test"]

X_train_processed_1 = prepared["experiments"][1]["X_train"]
X_valid_processed_1 = prepared["experiments"][1]["X_valid"]
X_test_processed_1 = prepared["experiments"][1]["X_test"]

X_train_processed_2 = prepared["experiments"][2]["X_train"]
X_valid_processed_2 = prepared["experiments"][2]["X_valid"]
X_test_processed_2 = prepared["experiments"][2]["X_test"]

X_train_processed_3 = prepared["experiments"][3]["X_train"]
X_valid_processed_3 = prepared["experiments"][3]["X_valid"]
X_test_processed_3 = prepared["experiments"][3]["X_test"]

for number, data in prepared["experiments"].items():
    print(f"Experiment {number}: train={data['X_train'].shape}, valid={data['X_valid'].shape}, test={data['X_test'].shape}")


## Export Datasets and Inspect Every DataFrame

The following cells save classification datasets under `Diamond/data/processed/classification/`
and display each DataFrame, its column types, missing values, and descriptive statistics.
`Source_Row` preserves the original CSV row position and `Split` identifies the shared
train/validation/test partition. Targets and metadata are excluded from model inputs.
These are descriptive summaries; model choice still uses validation Macro F1 only.


In [ ]:
from src.classification.datasets import export_datasets, show_dataset_info

DATASET_DIR = PROJECT_ROOT / "data" / "processed" / "classification"
dataset_files = export_datasets(prepared, DATASET_DIR)
display(dataset_files)
show_dataset_info(prepared)


## Training Feature Distributions and Correlations

Original exploration plots for all three experiments, using training rows only.


In [ ]:
from src.classification.diagnostics import show_diagnostics

show_diagnostics(prepared)


# Classification Experiments

Three feature experiments are evaluated using the same three classification models:

- Artificial Neural Network (ANN)
- Random Forest
- XGBoost

All experiments use the same stratified training, validation, and testing split.

The validation set is used to compare models and select the final model.

The test set remains untouched until the final model has been selected.

The primary model-selection metric is **Macro F1** because the clarity
families are not equally represented.

### Top-eight feature graphs

After each experiment, its highest validation Macro F1 model is used to rank
features by permutation importance: the decrease in Macro F1 when a feature is
shuffled. Five repeats use a reproducible stratified sample of up to 2,000
validation rows. The test set is not used.

The importance bar chart ranks the eight most influential features for that
model. The relationship plots show each numeric feature in its original units
across actual clarity families (**I, SI, VS, VVS, IF**); categorical features
show category proportions within each family. Error bars show variation across
shuffles. Correlated or engineered features can share importance; these plots
show predictive associations, not causal effects. A negative score drop means
shuffling did not harm performance in this check.


In [ ]:
# Evaluation is imported from src/classification/evaluate.py.

In [ ]:
# The maintained ANN architecture is imported from src/classification/model.py.

In [ ]:
# These are callbacks, setup for the model, so that they have with better results not noise
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=20,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=6,
    min_lr=1e-6
)

In [ ]:
# Storing the Results
results = []
trained_models = {}

## Experiment 1 — Physical Features

Experiment 1 evaluates whether diamond clarity can be predicted using
physical measurements, engineered physical features, cut, and color.

Price information is excluded.

### Models

- Artificial Neural Network
- Random Forest
- XGBoost

In [ ]:
ann_1 = build_ann(X_train_processed_1.shape[1])

history_ann_1 = ann_1.fit(
    X_train_processed_1,
    y_train,
    validation_data=(X_valid_processed_1, y_valid),
    epochs=160,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

ann_pred_1 = np.argmax(
    ann_1.predict(X_valid_processed_1, verbose=0),
    axis=1
)

results.append(
    evaluate_model(
        y_valid,
        ann_pred_1,
        "Experiment 1",
        "ANN"
    )
)

trained_models["Experiment 1 - ANN"] = {
    "model": ann_1,
    "experiment": 1,
    "type": "ANN"
}

In [ ]:
rf_1 = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_1.fit(
    X_train_processed_1,
    y_train
)

rf_pred_1 = rf_1.predict(
    X_valid_processed_1
)

results.append(
    evaluate_model(
        y_valid,
        rf_pred_1,
        "Experiment 1",
        "Random Forest"
    )
)

trained_models["Experiment 1 - Random Forest"] = {
    "model": rf_1,
    "experiment": 1,
    "type": "Tree"
}

In [ ]:
xgb_weights_1 = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb_1 = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    n_estimators=1800,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_1.fit(
    X_train_processed_1,
    y_train,
    sample_weight=xgb_weights_1,
    eval_set=[
        (X_valid_processed_1, y_valid)
    ],
    verbose=False
)

xgb_pred_1 = xgb_1.predict(
    X_valid_processed_1
)

results.append(
    evaluate_model(
        y_valid,
        xgb_pred_1,
        "Experiment 1",
        "XGBoost"
    )
)

trained_models["Experiment 1 - XGBoost"] = {
    "model": xgb_1,
    "experiment": 1,
    "type": "Tree"
}

In [ ]:
experiment_1_results = pd.DataFrame(results)

experiment_1_results = experiment_1_results[
    experiment_1_results["Experiment"] == "Experiment 1"
].copy()

display(
    experiment_1_results.sort_values(
        "Macro_F1",
        ascending=False
    )
)

In [ ]:
# Top Eight Features for Predicting Diamond Clarity
importance_1 = plot_top_features(prepared, trained_models, results, experiment_number=1)
display(importance_1.head(8))


## Experiment 2 — Physical Features + Raw Price

Experiment 2 adds the raw diamond price to the physical and categorical
features from Experiment 1.

The purpose is to determine whether price provides useful information
for distinguishing diamond clarity families.

### Models

- Artificial Neural Network
- Random Forest
- XGBoost

In [ ]:
ann_2 = build_ann(X_train_processed_2.shape[1])

history_ann_2 = ann_2.fit(
    X_train_processed_2,
    y_train,
    validation_data=(X_valid_processed_2, y_valid),
    epochs=160,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

ann_pred_2 = np.argmax(
    ann_2.predict(X_valid_processed_2, verbose=0),
    axis=1
)

results.append(
    evaluate_model(
        y_valid,
        ann_pred_2,
        "Experiment 2",
        "ANN"
    )
)

trained_models["Experiment 2 - ANN"] = {
    "model": ann_2,
    "experiment": 2,
    "type": "ANN"
}

In [ ]:
rf_2 = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_2.fit(
    X_train_processed_2,
    y_train
)

rf_pred_2 = rf_2.predict(
    X_valid_processed_2
)

results.append(
    evaluate_model(
        y_valid,
        rf_pred_2,
        "Experiment 2",
        "Random Forest"
    )
)

trained_models["Experiment 2 - Random Forest"] = {
    "model": rf_2,
    "experiment": 2,
    "type": "Tree"
}

In [ ]:
xgb_weights_2 = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb_2 = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    n_estimators=1800,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_2.fit(
    X_train_processed_2,
    y_train,
    sample_weight=xgb_weights_2,
    eval_set=[
        (X_valid_processed_2, y_valid)
    ],
    verbose=False
)

xgb_pred_2 = xgb_2.predict(
    X_valid_processed_2
)

results.append(
    evaluate_model(
        y_valid,
        xgb_pred_2,
        "Experiment 2",
        "XGBoost"
    )
)

trained_models["Experiment 2 - XGBoost"] = {
    "model": xgb_2,
    "experiment": 2,
    "type": "Tree"
}

In [ ]:
all_results = pd.DataFrame(results)

experiment_2_results = all_results[
    all_results["Experiment"] == "Experiment 2"
].copy()

display(
    experiment_2_results.sort_values(
        "Macro_F1",
        ascending=False
    )
)

In [ ]:
# Top Eight Features for Predicting Diamond Clarity
importance_2 = plot_top_features(prepared, trained_models, results, experiment_number=2)
display(importance_2.head(8))


## Experiment 3 — Physical + Price-Relative Features

Experiment 3 extends Experiment 2 by adding engineered price-relative
features.

These features measure price relative to different measurements of
diamond size.

### Additional Features

- Price per Carat
- Price per Volume
- Price per Face Area
- Price per Face Diagonal

### Models

- Artificial Neural Network
- Random Forest
- XGBoost

In [ ]:
ann_3 = build_ann(X_train_processed_3.shape[1])

history_ann_3 = ann_3.fit(
    X_train_processed_3,
    y_train,
    validation_data=(X_valid_processed_3, y_valid),
    epochs=160,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

ann_pred_3 = np.argmax(
    ann_3.predict(X_valid_processed_3, verbose=0),
    axis=1
)

results.append(
    evaluate_model(
        y_valid,
        ann_pred_3,
        "Experiment 3",
        "ANN"
    )
)

trained_models["Experiment 3 - ANN"] = {
    "model": ann_3,
    "experiment": 3,
    "type": "ANN"
}

In [ ]:
rf_3 = RandomForestClassifier(
    n_estimators=300,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_3.fit(
    X_train_processed_3,
    y_train
)

rf_pred_3 = rf_3.predict(
    X_valid_processed_3
)

results.append(
    evaluate_model(
        y_valid,
        rf_pred_3,
        "Experiment 3",
        "Random Forest"
    )
)

trained_models["Experiment 3 - Random Forest"] = {
    "model": rf_3,
    "experiment": 3,
    "type": "Tree"
}

In [ ]:
xgb_weights_3 = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb_3 = XGBClassifier(
    objective="multi:softprob",
    num_class=5,
    n_estimators=1800,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=4,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.05,
    reg_lambda=1.0,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=100,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

xgb_3.fit(
    X_train_processed_3,
    y_train,
    sample_weight=xgb_weights_3,
    eval_set=[
        (X_valid_processed_3, y_valid)
    ],
    verbose=False
)

xgb_pred_3 = xgb_3.predict(
    X_valid_processed_3
)

results.append(
    evaluate_model(
        y_valid,
        xgb_pred_3,
        "Experiment 3",
        "XGBoost"
    )
)

trained_models["Experiment 3 - XGBoost"] = {
    "model": xgb_3,
    "experiment": 3,
    "type": "Tree"
}

In [ ]:
all_results = pd.DataFrame(results)

experiment_3_results = all_results[
    all_results["Experiment"] == "Experiment 3"
].copy()

display(
    experiment_3_results.sort_values(
        "Macro_F1",
        ascending=False
    )
)

In [ ]:
# Top Eight Features for Predicting Diamond Clarity
importance_3 = plot_top_features(prepared, trained_models, results, experiment_number=3)
display(importance_3.head(8))


# Model Comparison and Evaluation

The validation results from all nine model configurations are compared.

The primary selection metric is **Macro F1** because it gives equal
importance to each clarity family regardless of class size.

Additional metrics are used to provide a broader view of model performance.

The strongest validation model is selected before the test set is used.

In [ ]:
# Overall Results
results_df = pd.DataFrame(results)

results_df["Run"] = (
    results_df["Experiment"]
    + " - "
    + results_df["Model"]
)

results_df = results_df.sort_values(
    "Macro_F1",
    ascending=False
).reset_index(drop=True)

display(results_df)

## Final Model Selection

The model with the highest validation Macro F1 score is selected as the
final classification model.

Only after the model has been selected is the untouched test set used
for final evaluation.

In [ ]:
best_row = results_df.iloc[0]

best_run = best_row["Run"]

print("Best Validation Model")
print("=" * 60)
print("Run:", best_run)
print("Experiment:", best_row["Experiment"])
print("Model:", best_row["Model"])
print(f"Validation Accuracy: {best_row['Accuracy']:.2f}%")
print(f"Validation Macro F1: {best_row['Macro_F1']:.2f}%")
print(
    f"Validation Balanced Accuracy: "
    f"{best_row['Balanced_Accuracy']:.2f}%"
)

In [ ]:
best_info = trained_models[best_run]

best_model = best_info["model"]
best_experiment = best_info["experiment"]
best_model_type = best_info["type"]

TEST_DATASETS = {
    1: X_test_processed_1,
    2: X_test_processed_2,
    3: X_test_processed_3
}

X_test_final = TEST_DATASETS[best_experiment]

In [ ]:
if best_model_type == "ANN":
    final_test_pred = np.argmax(
        best_model.predict(
            X_test_final,
            verbose=0
        ),
        axis=1
    )

else:
    final_test_pred = best_model.predict(
        X_test_final
    )

In [ ]:
final_test_results = evaluate_model(
    y_test,
    final_test_pred,
    f"Experiment {best_experiment}",
    best_row["Model"]
)

final_test_df = pd.DataFrame(
    [final_test_results]
)

display(final_test_df)

In [ ]:
print(
    classification_report(
        y_test,
        final_test_pred,
        target_names=CLARITY_CLASSES,
        digits=4,
        zero_division=0
    )
)

In [ ]:
cm = confusion_matrix(
    y_test,
    final_test_pred
)

cm_df = pd.DataFrame(
    cm,
    index=CLARITY_CLASSES,
    columns=CLARITY_CLASSES
)

display(cm_df)

In [ ]:
fig, ax = plt.subplots(
    figsize=(8, 7)
)

image = ax.imshow(cm)

ax.set_title(
    f"Final Test Confusion Matrix\n{best_run}"
)

ax.set_xlabel("Predicted Clarity Family")
ax.set_ylabel("Actual Clarity Family")

ax.set_xticks(
    np.arange(len(CLARITY_CLASSES))
)

ax.set_yticks(
    np.arange(len(CLARITY_CLASSES))
)

ax.set_xticklabels(CLARITY_CLASSES)
ax.set_yticklabels(CLARITY_CLASSES)

for i in range(len(CLARITY_CLASSES)):
    for j in range(len(CLARITY_CLASSES)):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

fig.colorbar(
    image,
    ax=ax
)

plt.tight_layout()
plt.show()

In [ ]:
validation_macro_f1 = best_row["Macro_F1"]
validation_accuracy = best_row["Accuracy"]

test_macro_f1 = final_test_results["Macro_F1"]
test_accuracy = final_test_results["Accuracy"]

final_comparison = pd.DataFrame({
    "Dataset": [
        "Validation",
        "Test"
    ],
    "Accuracy": [
        validation_accuracy,
        test_accuracy
    ],
    "Macro_F1": [
        validation_macro_f1,
        test_macro_f1
    ]
})

display(final_comparison)

## Classification Conclusion

Three feature configurations were evaluated using Artificial Neural Networks,
Random Forest, and XGBoost.

Experiment 1 measured performance using physical and categorical information
without price.

Experiment 2 investigated whether adding raw diamond price improved clarity
classification.

Experiment 3 investigated whether price-relative engineered features provided
additional predictive information.

The models were compared using the validation set, with Macro F1 used as the
primary model-selection metric.

The strongest validation model was selected and evaluated once on the
previously untouched test set.

The final test results provide an estimate of how well the selected model
generalizes to unseen diamond observations.

## Save and Use the Best Model

The model selected by validation Macro F1 is saved with its fitted preprocessor to
`Diamond/models/classification/`. This preserves the exact model evaluated above.
New predictions use raw diamond inputs; do not supply the unknown clarity or engineered
features. Price is required when experiment 2 or 3 wins. `GET /model` lists required inputs.
Class probabilities are model estimates, not calibrated guarantees.


In [ ]:
from src.classification.prediction import save_best_model, load_best_model, predict_diamonds

MODEL_DIR = PROJECT_ROOT / "models" / "classification"
save_best_model(
    best_model, best_model_type,
    prepared["experiments"][best_experiment]["preprocessor"],
    best_experiment, best_run, best_row.to_dict(), MODEL_DIR
)
best_bundle = load_best_model(MODEL_DIR)
print("Saved model:", best_bundle["metadata"]["model_name"])
print("Required inputs:", best_bundle["metadata"]["required_inputs"])


In [ ]:
# Edit These Raw Diamond Details to Try Your Own Prediction
example_diamond = {
    "carat": 0.70, "cut": "Ideal", "color": "G", "depth": 61.5,
    "table": 57.0, "x": 5.70, "y": 5.72, "z": 3.51, "price": 2500.0
}
display(pd.DataFrame(predict_diamonds(best_bundle, example_diamond)))


## Local Prediction API

The next cell starts `http://127.0.0.1:8765` without blocking the notebook.
Keep the kernel running while using this API. Send one diamond JSON object or a list
of up to 1,000 objects to `POST /predict`; inspect required fields at `GET /model`.
Rerunning the cell replaces only the API previously started by this notebook.

To run the saved model after closing the notebook, open a terminal in `Diamond` and run:

```bash
python -m src.classification.api --port 8765
```

Use the same Python environment that trained the model. The local API is available
on this computer only. To stop the notebook API, run
`prediction_server.shutdown()` followed by `prediction_server.server_close()`.


In [ ]:
from src.classification.api import start_prediction_api

if "prediction_server" in globals():
    prediction_server.shutdown()
    prediction_server.server_close()
prediction_server = start_prediction_api(MODEL_DIR, port=8765)


In [ ]:
# Send a Real HTTP Request to Your Best Model
import json
from urllib.request import Request, urlopen

request = Request(
    "http://127.0.0.1:8765/predict",
    data=json.dumps(example_diamond).encode("utf-8"),
    headers={"Content-Type": "application/json"}, method="POST"
)
with urlopen(request, timeout=30) as response:
    display(json.loads(response.read()))
